# Phase 3 — Recommendation Engine & Model Comparison

We compare 4 KNN approaches (all use `NearestNeighbors(metric='cosine')`):

| Approach | Features |
|---|---|
| A | Ingredients only (multi-hot) |
| B | Ingredients + family + subfamily + gender (OHE) |
| C | B + TF-IDF on description |
| D | Weighted: ingredients × 2 + categories |

Evaluation metrics (no ratings needed):
- **Ingredient Overlap Score** — do results share notes with the query?
- **Family Match Rate** — % of results matching the query's family
- **Intra-List Diversity (ILD)** — how varied are the 10 results?

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib
import os
import time
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_distances
import matplotlib.pyplot as plt
import seaborn as sns

MODELS_DIR = '../models'

df          = joblib.load(f'{MODELS_DIR}/perfume_df.pkl')
mlb         = joblib.load(f'{MODELS_DIR}/mlb_ingredients.pkl')
ohe         = joblib.load(f'{MODELS_DIR}/ohe_categories.pkl')
tfidf       = joblib.load(f'{MODELS_DIR}/tfidf_description.pkl')

matrix_A = load_npz(f'{MODELS_DIR}/matrix_A.npz')
matrix_B = load_npz(f'{MODELS_DIR}/matrix_B.npz')
matrix_C = load_npz(f'{MODELS_DIR}/matrix_C.npz')
matrix_D = load_npz(f'{MODELS_DIR}/matrix_D.npz')

print(f'Loaded {len(df):,} perfumes')

## Step 1 — Fit KNN for Each Approach

In [ ]:
approaches = {
    'A': matrix_A,
    'B': matrix_B,
    'C': matrix_C,
    'D': matrix_D,
}

knn_models = {}
for name, matrix in approaches.items():
    t0 = time.time()
    knn = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute', n_jobs=-1)
    knn.fit(matrix)
    elapsed = time.time() - t0
    knn_models[name] = knn
    print(f'Approach {name}: fitted in {elapsed:.2f}s  |  matrix shape: {matrix.shape}')

## Step 2 — Build Query Vector Helper

In [ ]:
from scipy.sparse import hstack, csr_matrix

def build_query_vector(approach, liked_ingredients, family, subfamily, gender, description=''):
    """
    Encode user preferences into a query vector matching the chosen approach's feature space.
    """
    # Ingredients multi-hot
    ing_vec = mlb.transform([liked_ingredients])  # shape (1, n_ingredients)
    
    if approach == 'A':
        return ing_vec
    
    # Categories OHE
    cat_input = pd.DataFrame([[family, subfamily, gender]],
                              columns=['family', 'subfamily', 'gender'])
    cat_vec = ohe.transform(cat_input)  # shape (1, n_categories)
    
    if approach == 'B':
        return hstack([ing_vec, cat_vec])
    
    if approach == 'C':
        desc_vec = tfidf.transform([description])
        return hstack([ing_vec, cat_vec, desc_vec])
    
    if approach == 'D':
        return hstack([ing_vec * 2, cat_vec])
    
    raise ValueError(f'Unknown approach: {approach}')


def recommend(approach, liked_ingredients, family='UNKNOWN', subfamily='UNKNOWN',
              gender='UNKNOWN', description='', n=10, gender_filter=True):
    """
    Return top-N recommendations for the given preferences.
    """
    matrix = approaches[approach]
    knn = knn_models[approach]
    
    # Optional: pre-filter by gender to focus results
    if gender_filter and gender != 'UNKNOWN':
        mask = (df['gender'] == gender) | (df['gender'] == 'UNISEX')
        filtered_df = df[mask].reset_index(drop=True)
        filtered_matrix = matrix[mask.values]
        knn_local = NearestNeighbors(n_neighbors=min(n+1, len(filtered_df)),
                                     metric='cosine', algorithm='brute', n_jobs=-1)
        knn_local.fit(filtered_matrix)
    else:
        filtered_df = df
        filtered_matrix = matrix
        knn_local = knn
    
    query = build_query_vector(approach, liked_ingredients, family, subfamily, gender, description)
    distances, indices = knn_local.kneighbors(query, n_neighbors=min(n+1, len(filtered_df)))
    
    results = filtered_df.iloc[indices[0]].copy()
    results['similarity'] = (1 - distances[0]).round(4)
    return results.head(n)

print('Helper functions defined.')

## Step 3 — Define Evaluation Metrics

In [ ]:
def ingredient_overlap_score(results_df, query_ingredients):
    """Average Jaccard similarity between query notes and each result's notes."""
    query_set = set(q.title() for q in query_ingredients)
    scores = []
    for ing_list in results_df['ingredients']:
        result_set = set(ing_list)
        if not query_set or not result_set:
            scores.append(0.0)
            continue
        intersection = len(query_set & result_set)
        union = len(query_set | result_set)
        scores.append(intersection / union)
    return np.mean(scores)


def family_match_rate(results_df, query_family):
    """Fraction of results that share the same family as the query."""
    if query_family == 'UNKNOWN':
        return np.nan
    return (results_df['family'] == query_family).mean()


def intra_list_diversity(results_df, approach):
    """Average pairwise cosine distance among the top-N results. Higher = more diverse."""
    matrix = approaches[approach]
    indices = results_df.index.tolist()
    if len(indices) < 2:
        return 0.0
    sub = matrix[indices]
    dists = cosine_distances(sub)
    n = len(indices)
    upper = dists[np.triu_indices(n, k=1)]
    return upper.mean()


print('Evaluation functions defined.')

## Step 4 — Run Comparison on 5 Test Queries

In [ ]:
test_queries = [
    dict(liked_ingredients=['Rose', 'Jasmine', 'Bergamot'],
         family='FLORAL', subfamily='FLORAL SOFT', gender='FEMALE',
         description='romantic floral feminine fragrance'),
    dict(liked_ingredients=['Oud', 'Sandalwood', 'Amber'],
         family='WOODY', subfamily='AMBERY (ORIENTAL)', gender='MALE',
         description='rich woody oriental masculine scent'),
    dict(liked_ingredients=['Citrus', 'Lemon', 'Mint', 'Bergamot'],
         family='FRESH', subfamily='CITRUS', gender='MALE',
         description='fresh citrus summer fragrance'),
    dict(liked_ingredients=['Vanilla', 'Musk', 'Sandalwood'],
         family='ORIENTAL', subfamily='SOFT ORIENTAL', gender='FEMALE',
         description='warm sweet vanilla musk'),
    dict(liked_ingredients=['Pepper', 'Leather', 'Vetiver'],
         family='WOODY', subfamily='WOODY AROMATIC', gender='MALE',
         description='spicy leather dark masculine'),
]

results_table = []

for approach in ['A', 'B', 'C', 'D']:
    ilo_scores, fmr_scores, ild_scores = [], [], []
    for q in test_queries:
        res = recommend(
            approach,
            liked_ingredients=q['liked_ingredients'],
            family=q['family'],
            subfamily=q['subfamily'],
            gender=q['gender'],
            description=q.get('description', ''),
            n=10
        )
        ilo_scores.append(ingredient_overlap_score(res, q['liked_ingredients']))
        fmr_scores.append(family_match_rate(res, q['family']))
        ild_scores.append(intra_list_diversity(res, approach))
    
    results_table.append({
        'Approach': approach,
        'Ingredient Overlap (↑)': round(np.mean(ilo_scores), 4),
        'Family Match Rate (↑)':  round(np.nanmean(fmr_scores), 4),
        'Intra-List Diversity (↑)': round(np.mean(ild_scores), 4),
    })

comparison_df = pd.DataFrame(results_table)
comparison_df['Score (avg of 3)'] = comparison_df[[
    'Ingredient Overlap (↑)', 'Family Match Rate (↑)', 'Intra-List Diversity (↑)'
]].mean(axis=1).round(4)

comparison_df = comparison_df.sort_values('Score (avg of 3)', ascending=False).reset_index(drop=True)
print(comparison_df.to_string(index=False))

## Step 5 — Visualise Comparison

In [ ]:
metrics = ['Ingredient Overlap (↑)', 'Family Match Rate (↑)', 'Intra-List Diversity (↑)']
x = np.arange(len(metrics))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['steelblue', 'tomato', 'seagreen', 'goldenrod']

for i, row in comparison_df.iterrows():
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=f'Approach {row["Approach"]}', color=colors[i])

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['Ingredient\nOverlap', 'Family\nMatch Rate', 'Intra-List\nDiversity'])
ax.set_ylabel('Score')
ax.set_title('Model Comparison — 4 KNN Approaches', fontweight='bold', fontsize=13)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print(comparison_df.to_string(index=False))

## Step 6 — Save the Best Model

In [ ]:
best_approach = comparison_df.iloc[0]['Approach']
print(f'Best approach: {best_approach}')

# Refit on full dataset (n_neighbors=11 to skip self-match)
best_matrix = approaches[best_approach]
knn_best = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute', n_jobs=-1)
knn_best.fit(best_matrix)

joblib.dump(knn_best, f'{MODELS_DIR}/knn_model.pkl')
joblib.dump(best_approach, f'{MODELS_DIR}/best_approach.pkl')
comparison_df.to_csv(f'{MODELS_DIR}/model_comparison.csv', index=False)

print(f'Saved knn_model.pkl (approach {best_approach})')
print('Saved model_comparison.csv')

## Step 7 — Sanity Check: Query Known Perfume

In [ ]:
# Pick a well-known perfume and verify its top match is itself
sample = df[df['name_perfume'].str.contains('Chanel', case=False, na=False)].iloc[0]
print(f'Sanity query: {sample["name_perfume"]} by {sample["brand"]}')
print(f'Ingredients: {sample["ingredients"][:6]}')
print(f'Family: {sample["family"]} | Gender: {sample["gender"]}\n')

top = recommend(
    best_approach,
    liked_ingredients=sample['ingredients'],
    family=sample['family'],
    subfamily=sample['subfamily'],
    gender=sample['gender'],
    n=5
)
print('Top 5 similar perfumes:')
print(top[['name_perfume', 'brand', 'family', 'similarity']].to_string(index=False))